# OpenPublicDomainFootballData

In [139]:
import pandas as pd
import glob
import os

RAW = 'OpenPublicDomainFootballData'

archivos_mx = sorted(glob.glob(os.path.join(RAW, '*', 'mx.1.csv')))
print(f"Archivos encontrados: {len(archivos_mx)}")

df_opfd = pd.concat(
    [pd.read_csv(f) for f in archivos_mx],
    ignore_index=True
)
print(f"Shape: {df_opfd.shape}")

Archivos encontrados: 7
Shape: (2191, 13)


In [140]:
display(df_opfd.head())

,Stage,Round,Date,Time,Timezone,Team 1,FT,HT,Team 2,UTC,ET,P,Comments
0,Apertura,1,Fri Jul 20 2018,21:00,CDT/-0500,Atlas Guadalajara,0-0,0-0,Gallos Blancos,2018-07-21T02:00Z,NaN,NaN,NaN
1,Apertura,1,Fri Jul 20 2018,19:00,CDT/-0500,CD Veracruz,0-2,0-2,Pumas UNAM,2018-07-21T00:00Z,NaN,NaN,NaN
2,Apertura,1,Sat Jul 21 2018,17:00,CDT/-0500,Cruz Azul,3-0,0-0,Puebla FC,2018-07-21T22:00Z,NaN,NaN,NaN
3,Apertura,1,Sat Jul 21 2018,19:00,CDT/-0500,CF Pachuca,0-1,0-0,CF Monterrey,2018-07-22T00:00Z,NaN,NaN,NaN
4,Apertura,1,Sat Jul 21 2018,21:00,CDT/-0500,UANL Tigres,2-0,1-0,Club León,2018-07-22T02:00Z,NaN,NaN,NaN


In [141]:
# Para ver todos los valores no numéricos en FT y HT
import re

def es_marcador(val):
    if pd.isnull(val):
        return True
    return bool(re.match(r'^\d+-\d+$', str(val).strip()))

print("Valores raros en FT:")
print(df_opfd[~df_opfd['FT'].apply(es_marcador)]['FT'].value_counts())

print("\nValores raros en HT:")
print(df_opfd[~df_opfd['HT'].apply(es_marcador)]['HT'].value_counts())

Valores raros en FT:
FT
(*)        63
3-0 (*)     1
0-3 (*)     1
Name: count, dtype: int64

Valores raros en HT:
Series([], Name: count, dtype: int64)


In [142]:
# Deduplicar por la combinación fecha + equipos (llave natural del partido)
antes = len(df_opfd)
df_opfd = df_opfd.drop_duplicates(subset=['Date', 'Team 1', 'Team 2'])
print(f"Duplicados eliminados: {antes - len(df_opfd)}")

Duplicados eliminados: 0


In [143]:
display(df_opfd.isnull().sum()) # Porque se me olvidaron los nulos

Stage          0
Round          0
Date           0
Time          63
Timezone      63
Team 1         0
FT            79
HT           144
Team 2         0
UTC           63
ET          2186
P           2179
Comments    2126
dtype: int64

In [144]:
df_opfd['Comments'].value_counts() # Para conocer cuantos partidos hay raros

Comments
cancelled    63
awd.          2
Name: count, dtype: int64

In [145]:
# Nos quedamos solo con los partidos que no tengan comentarios
df_jugados = df_opfd[df_opfd['Comments'].isnull()]
display(df_jugados)

,Stage,Round,Date,Time,Timezone,Team 1,FT,HT,Team 2,UTC,ET,P,Comments
0,Apertura,1,Fri Jul 20 2018,21:00,CDT/-0500,Atlas Guadalajara,0-0,0-0,Gallos Blancos,2018-07-21T02:00Z,NaN,NaN,NaN
1,Apertura,1,Fri Jul 20 2018,19:00,CDT/-0500,CD Veracruz,0-2,0-2,Pumas UNAM,2018-07-21T00:00Z,NaN,NaN,NaN
2,Apertura,1,Sat Jul 21 2018,17:00,CDT/-0500,Cruz Azul,3-0,0-0,Puebla FC,2018-07-21T22:00Z,NaN,NaN,NaN
3,Apertura,1,Sat Jul 21 2018,19:00,CDT/-0500,CF Pachuca,0-1,0-0,CF Monterrey,2018-07-22T00:00Z,NaN,NaN,NaN
4,Apertura,1,Sat Jul 21 2018,21:00,CDT/-0500,UANL Tigres,2-0,1-0,Club León,2018-07-22T02:00Z,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,Apertura,17,Sat Nov 9 2024,19:00,CST/-0600,Deportivo Toluca,NaN,NaN,CF América,2024-11-10T01:00Z,NaN,NaN,NaN
2187,Apertura,17,Sat Nov 9 2024,21:00,CST/-0600,Cruz Azul,NaN,NaN,UANL Tigres,2024-11-10T03:00Z,NaN,NaN,NaN
2188,Apertura,17,Sun Nov 10 2024,19:00,CST/-0600,CF Monterrey,NaN,NaN,Club León,2024-11-11T01:00Z,NaN,NaN,NaN
2189,Apertura,17,Sun Nov 10 2024,21:00,CST/-0600,Club Tijuana,NaN,NaN,Puebla FC,2024-11-11T03:00Z,NaN,NaN,NaN


In [146]:
import unicodedata

# Normalizamos las columnas string
def normalizar_strings(df):
    cols_str = df.select_dtypes(include='object').columns
    for col in cols_str:
        df[col] = (df[col]
                   .astype(str)                             
                   .str.lower()
                   .str.normalize('NFKD')
                   .str.encode('ascii', errors='ignore')
                   .str.decode('utf-8')
                   .str.strip()
                   .replace('nan', pd.NA))                        
    return df

df_jugados = normalizar_strings(df_jugados)

C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\2786774112.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = (df[col]
C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\2786774112.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = (df[col]
C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\2786774112.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se

In [147]:
# Homogeneizamos formatos
df_jugados['Date'] = pd.to_datetime(df_jugados['Date'], dayfirst=True, errors='coerce')
# Revisamos si quedaron fechas inválidas
print(f"Fechas nulas tras parseo: {df_jugados['Date'].isnull().sum()}")

Fechas nulas tras parseo: 0


C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\4200374456.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_jugados['Date'] = pd.to_datetime(df_jugados['Date'], dayfirst=True, errors='coerce')
C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\4200374456.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jugados['Date'] = pd.to_datetime(df_jugados['Date'], dayfirst=True, errors='coerce')


In [148]:
# Ver todos los valores no numéricos en FT y HT
import re

def es_marcador(val):
    if pd.isnull(val):
        return True
    return bool(re.match(r'^\d+-\d+$', str(val).strip()))

print("Valores raros en FT:")
print(df_jugados[~df_jugados['FT'].apply(es_marcador)]['FT'].value_counts())

print("\nValores raros en HT:")
print(df_jugados[~df_jugados['HT'].apply(es_marcador)]['HT'].value_counts())

Valores raros en FT:
Series([], Name: count, dtype: int64)

Valores raros en HT:
Series([], Name: count, dtype: int64)


In [149]:
# Por algun motivo aun habia (*), así que los eliminamos a la fuerza
def split_marcador(serie):
    limpia = serie.str.strip()
    limpia = limpia.replace('(*)', None)  # tratar (*) como nulo
    limpia = limpia.where(limpia.str.match(r'^\d+-\d+$', na=False), other=None)
    return limpia.str.split('-', expand=True).astype('Int64')

df_jugados[['FT1', 'FT2']] = split_marcador(df_jugados['FT'])
df_jugados[['HT1', 'HT2']] = split_marcador(df_jugados['HT'])

C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\503254044.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jugados[['FT1', 'FT2']] = split_marcador(df_jugados['FT'])
C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\503254044.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jugados[['FT1', 'FT2']] = split_marcador(df_jugados['FT'])
C:\Users\TEMP.LABLCD.002\AppData\Local\Temp\ipykernel_23900\503254044.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [150]:
# Finalmente, nos deshacemos de las columnas de puntaje en tiempo extra y penales
df_csvs_limpio = df_jugados.drop(['ET', 'P', 'Comments', 'FT', 'HT'], axis = 1)

display(df_csvs_limpio.isnull().sum()) # Veremos si obtenemos los puntajes faltantes en el record linkage

Stage        0
Round        0
Date         0
Time         0
Timezone     0
Team 1       0
Team 2       0
UTC          0
FT1         79
FT2         79
HT1         79
HT2         79
dtype: int64

In [151]:
display(df_csvs_limpio.head())

,Stage,Round,Date,Time,Timezone,Team 1,Team 2,UTC,FT1,FT2,HT1,HT2
0,apertura,1,2018-07-20,21:00,cdt/-0500,atlas guadalajara,gallos blancos,2018-07-21t02:00z,0,0,0,0
1,apertura,1,2018-07-20,19:00,cdt/-0500,cd veracruz,pumas unam,2018-07-21t00:00z,0,2,0,2
2,apertura,1,2018-07-21,17:00,cdt/-0500,cruz azul,puebla fc,2018-07-21t22:00z,3,0,0,0
3,apertura,1,2018-07-21,19:00,cdt/-0500,cf pachuca,cf monterrey,2018-07-22t00:00z,0,1,0,0
4,apertura,1,2018-07-21,21:00,cdt/-0500,uanl tigres,club leon,2018-07-22t02:00z,2,0,1,0


# LigaMX

In [152]:
df_ligamx = pd.read_json('LigaMX/2016-2024_liga_mx.json')
print(df_ligamx.shape)
print(df_ligamx.columns)
display(df_ligamx.head())

(2876, 23)
Index(['id', 'referee', 'timezone', 'date', 'venue_id', 'venue_name',
       'venue_city', 'season', 'round', 'home_team', 'away_team', 'home_win',
       'away_win', 'home_goals', 'away_goals', 'home_goals_half_time',
       'away_goals_half_time', 'home_goals_fulltime', 'away_goals_fulltime',
       'home_goals_extra_time', 'away_goals_extratime', 'home_goals_penalty',
       'away_goals_penalty'],
      dtype='object')


,id,referee,timezone,date,venue_id,venue_name,venue_city,season,round,home_team,...,home_goals,away_goals,home_goals_half_time,away_goals_half_time,home_goals_fulltime,away_goals_fulltime,home_goals_extra_time,away_goals_extratime,home_goals_penalty,away_goals_penalty
0,864095,D. Quintero,UTC,2022-07-02 02:05:00+00:00,10546.0,Estadio de Mazatlán,Mazatlán,2022,Apertura - 1,Mazatlán,...,2.0,4.0,0.0,2.0,2.0,4.0,NaN,NaN,NaN,NaN
1,864096,I. Lopez,UTC,2022-07-02 22:00:00+00:00,1076.0,Estadio AKRON,Zapopan,2022,Apertura - 1,Guadalajara Chivas,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,864094,L. Santander,UTC,2022-07-02 22:00:00+00:00,1080.0,Estadio Victoria de Aguascalientes,Aguascalientes,2022,Apertura - 1,Necaxa,...,1.0,3.0,1.0,2.0,1.0,3.0,NaN,NaN,NaN,NaN
3,864097,Ó. Mejía,UTC,2022-07-03 00:05:00+00:00,1087.0,Estadio Universitario de Nuevo León,San Nicolás de los Garza,2022,Apertura - 1,Tigres UANL,...,2.0,3.0,0.0,1.0,2.0,3.0,NaN,NaN,NaN,NaN
4,864098,F. Guerrero,UTC,2022-07-03 02:05:00+00:00,7182.0,Estadio Azteca,D.F.,2022,Apertura - 1,Club America,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [153]:
df_ligamx.isnull().sum()

id                          0
referee                   271
timezone                    0
date                        0
venue_id                  900
venue_name                  0
venue_city                181
season                      0
round                       0
home_team                   0
away_team                   0
home_win                  810
away_win                  810
home_goals                 63
away_goals                 63
home_goals_half_time       63
away_goals_half_time       63
home_goals_fulltime        63
away_goals_fulltime        63
home_goals_extra_time    2860
away_goals_extratime     2860
home_goals_penalty       2862
away_goals_penalty       2862
dtype: int64

In [154]:
# orregir home_win / away_win cuando son null por empate
df_ligamx['draw'] = (df_ligamx['home_goals'] == df_ligamx['away_goals'])

df_ligamx['home_win'] = df_ligamx['home_win'].fillna(
    df_ligamx['home_goals'] > df_ligamx['away_goals']
)
df_ligamx['away_win'] = df_ligamx['away_win'].fillna(
    df_ligamx['away_goals'] > df_ligamx['home_goals']
)

# Verificar que ya no haya nulls donde había partido jugado
print(df_ligamx[['home_win', 'away_win', 'draw']].isnull().sum())

home_win    0
away_win    0
draw        0
dtype: int64


In [155]:
# Parsear fecha
df_ligamx['date'] = pd.to_datetime(df_ligamx['date'], dayfirst=True, errors='coerce').dt.normalize().dt.tz_localize(None)

In [156]:
# Goles de float a Int64 (mayúscula — soporta NaN, int normal no)
cols_goles = [c for c in df_ligamx.columns if 'goal' in c]
df_ligamx[cols_goles] = df_ligamx[cols_goles].astype('Int64')

In [157]:
# Corregir typo en nombre de columna
df_ligamx = df_ligamx.rename(columns={
    'away_goals_extratime': 'away_goals_extra_time'
})

In [158]:
# Normalizamos las columnas string
df_ligamx = normalizar_strings(df_ligamx)

In [162]:
df_jsons_limpio = df_ligamx.drop('id', axis = 1)
display(df_jsons_limpio)

,referee,timezone,date,venue_id,venue_name,venue_city,season,round,home_team,away_team,...,away_goals,home_goals_half_time,away_goals_half_time,home_goals_fulltime,away_goals_fulltime,home_goals_extra_time,away_goals_extra_time,home_goals_penalty,away_goals_penalty,draw
0,d. quintero,utc,2022-07-02,10546.0,estadio de mazatlan,mazatlan,2022,apertura - 1,mazatlan,puebla,...,4,0,2,2,4,<NA>,<NA>,<NA>,<NA>,False
1,i. lopez,utc,2022-07-02,1076.0,estadio akron,zapopan,2022,apertura - 1,guadalajara chivas,fc juarez,...,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,True
2,l. santander,utc,2022-07-02,1080.0,estadio victoria de aguascalientes,aguascalientes,2022,apertura - 1,necaxa,toluca,...,3,1,2,1,3,<NA>,<NA>,<NA>,<NA>,False
3,o. mejia,utc,2022-07-03,1087.0,estadio universitario de nuevo leon,san nicolas de los garza,2022,apertura - 1,tigres uanl,cruz azul,...,3,0,1,2,3,<NA>,<NA>,<NA>,<NA>,False
4,f. guerrero,utc,2022-07-03,7182.0,estadio azteca,d.f.,2022,apertura - 1,club america,atlas,...,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2871,"marco antonio ortiz nava, mexico",utc,2017-05-19,NaN,estadio universitario (uanl),monterrey,2016,clausura - semi-finals,tigres uanl,club tijuana,...,0,2,0,2,0,<NA>,<NA>,<NA>,<NA>,False
2872,"luis santander, mexico",utc,2017-05-21,NaN,estadio akron,guadalajara,2016,clausura - semi-finals,guadalajara chivas,toluca,...,1,1,0,1,1,<NA>,<NA>,<NA>,<NA>,True
2873,"jorge antonio perez duran, mexico",utc,2017-05-22,1088.0,estadio caliente,tijuana,2016,clausura - semi-finals,club tijuana,tigres uanl,...,2,0,0,0,2,<NA>,<NA>,<NA>,<NA>,False
2874,"oscar macias, mexico",utc,2017-05-26,NaN,estadio universitario (uanl),monterrey,2016,clausura - finals,tigres uanl,guadalajara chivas,...,2,0,2,2,2,<NA>,<NA>,<NA>,<NA>,True
